## Vaults for Secret Storage with Claude Managed Agents

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")
environment_id = os.getenv("ENVIRONMENT_ID")
github_pat = os.getenv("GITHUB_PAT")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=claude_api_key)

### Create the Vault

In [ ]:
vault = client.beta.vaults.create(
    display_name="GitHub PAT Vault",
    metadata={
        "github_username": "ENTER-YOUR-GITHUB-USERNAME-HERE"
    }
)

print(vault.id)

### Register the GitHub PAT Secret in the Vault

In [ ]:
credential = client.beta.vaults.credentials.create(

    vault_id=vault.id,

    display_name="GitHub PAT",

    auth={
        "type": "static_bearer",

        "mcp_server_url": "https://api.githubcopilot.com/mcp/",

        "token": github_pat
    }
)

### Create the Agent

In [ ]:
agent = client.beta.agents.create(

    name="GitHub Assistant",

    model=claude_model_name,

    system="""
You are an expert GitHub assistant.
""",

    mcp_servers=[
        {
            "type": "url",
            "name": "GitHub",
            "url": "https://api.githubcopilot.com/mcp/"
        }
    ],

    tools=[
        {
            "type": "agent_toolset_20260401"
        },
        {
            "type": "mcp_toolset",
            "mcp_server_name": "GitHub",
            "default_config": {
                "permission_policy": {
                    "type": "always_allow"
                }
            }
        }
    ]
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

### Create a Session

In [ ]:
session_one = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment_id,
    vault_ids=[vault.id]
)

### Execute the Agent

In [ ]:
with client.beta.sessions.events.stream(session_one.id) as stream:
            # Send the user message after the stream opens
            client.beta.sessions.events.send(
                session_one.id,
                events=[
                    {
                        "type": "user.message",
                        "content": [
                            {
                                "type": "text",
                                "text": """Tell me contents included in the GitHub Repo - kuljotSB/Claude-Certified-Developer-CCDV-F""",
                            },
                        ],
                    },
                ],
            )

            # Process streaming events
            for event in stream:
                match event.type:
                    case "agent.message":
                        for block in event.content:
                            print(block.text, end="")
                    case "agent.tool_use":
                        print(f"\n[Using tool: {event.name}]")
                    case "session.status_idle":
                        print("\n\nAgent finished.")
                        break